## modelling

In [ ]:
!pip install pandas -q 
!pip install scikit-learn -q 

In [ ]:
# =========================================================
# System-level utilities for process control and execution
# =========================================================
import os
import time
import psutil

# =========================================================
# Core numerical computation and data management
# =========================================================
import numpy as np
import pandas as pd

# =========================================================
# Regression models for age prediction and bias correction
# =========================================================
from sklearn.svm import LinearSVR
from sklearn.linear_model import LinearRegression

# =========================================================
# Cross-validation and performance assessment
# =========================================================
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

# =========================================================
# Feature scaling and normalization
# =========================================================
from sklearn.preprocessing import StandardScaler

# =========================================================
# Model serialization and parallel execution
# =========================================================
import joblib
from joblib import Parallel, delayed


In [ ]:
# =========================================================
# Computational resource control
# =========================================================
def set_low_cpu_priority():
    """Reduce CPU scheduling priority of the current process."""
    pid = os.getpid()
    p = psutil.Process(pid)
    p.nice(19)

set_low_cpu_priority()

# =========================================================
# Data loading and feature definition
# =========================================================
data = pd.read_csv('./input/train_white_matter.csv')
print(data.columns)

X = data.drop(columns=['age', 'sex', "Unnamed: 0", "eid"]).values
age = data['age'].values
sex = data['sex'].values

# Sex-stratified analysis (0 = female, 1 = male)
Gender = 0
X = X[sex == Gender]
age = age[sex == Gender]

# =========================================================
# Cross-validation design
# =========================================================
K = 20
kf = KFold(n_splits=K, shuffle=True, random_state=42)
age_predic = np.zeros(len(age))

def train_fold(k, train_index, test_index, X, age):
    """Train and evaluate LinearSVR within one cross-validation fold."""
    xTrain, xTest = X[train_index], X[test_index]
    yTrain, yTest = age[train_index], age[test_index]

    # Feature standardization performed within each fold
    scaler = StandardScaler()
    xTrain_scaled = scaler.fit_transform(xTrain)
    xTest_scaled = scaler.transform(xTest)

    print(f'Training fold {k + 1}...')

    model = LinearSVR(C=1.0, epsilon=0.1, max_iter=10000)
    model.fit(xTrain_scaled, yTrain)

    yhat = model.predict(xTest_scaled)
    return test_index, yhat

# =========================================================
# Parallelized cross-validation training
# =========================================================
num_jobs = 4
results = Parallel(n_jobs=num_jobs)(
    delayed(train_fold)(k, train_index, test_index, X, age)
    for k, (train_index, test_index) in enumerate(kf.split(X))
)

for test_index, yhat in results:
    age_predic[test_index] = yhat

print("K-fold training completed!")

# =========================================================
# Model performance evaluation
# =========================================================
r_predic, _ = pearsonr(age, age_predic)
mae = mean_absolute_error(age, age_predic)
print(f'Prediction outcome: correlation r={r_predic:.2f}, MAE={mae:.2f}')

# =========================================================
# Age-related prediction bias correction
# =========================================================
gap = age_predic - age

beta_model = LinearRegression()
beta_model.fit(age.reshape(-1, 1), gap)

gap_corrected = gap - beta_model.predict(age.reshape(-1, 1))

# =========================================================
# Final model training on the full dataset
# =========================================================
scaler_final = StandardScaler()
X_scaled = scaler_final.fit_transform(X)

final_model = LinearSVR(C=1.0, epsilon=0.1, max_iter=10000)
final_model.fit(X_scaled, age)

print(f'Final model trained on full dataset (n={len(age)})')

# =========================================================
# Model persistence
# =========================================================
save_path = f'model_sex{Gender}.pkl'

joblib.dump({
    'model': final_model,
    'scaler': scaler_final,
    'age_real': age,
    'age_predic': age_predic,
    'r_predic': r_predic,
    'mae': mae,
    'sex': Gender,
    'beta_model': beta_model
}, save_path)

print(f'Model and results saved to {save_path}')


## test model

In [ ]:
# =========================================================
# Core numerical and scientific computing libraries
# =========================================================
import numpy as np
import scipy.io as sio

# =========================================================
# Feature preprocessing and evaluation metrics
# =========================================================
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# =========================================================
# Model persistence and system utilities
# =========================================================
import joblib
import os
import time
import psutil

# =========================================================
# Data handling
# =========================================================
import numpy as np
import pandas as pd

# =========================================================
# Machine learning models and validation
# =========================================================
from sklearn.svm import LinearSVR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# =========================================================
# Statistical analysis
# =========================================================
from scipy.stats import pearsonr

# =========================================================
# Parallel computation
# =========================================================
import joblib
from joblib import Parallel, delayed


In [ ]:
# =========================================================
# Load independent test dataset
# =========================================================
data = pd.read_csv('./input/test_white_matter.csv')

# Inspect available variables in the test dataset
print(data.columns)

# =========================================================
# Construct feature matrix and target variables
# =========================================================
# Extract imaging-derived features by removing non-feature columns
X_test = data.drop(columns=['age', 'sex','Unnamed: 0', 'eid']).values

# Extract chronological age as the prediction target
age_test = data['age'].values

# Extract biological sex for stratified evaluation
sex_test = data['sex'].values

# =========================================================
# Sex-stratified test sample selection
# =========================================================
Gender = 0  # 0 = female, 1 = male
X_test = X_test[sex_test == Gender, :]
age_test = age_test[sex_test == Gender]

# =========================================================
# Load trained model and preprocessing parameters
# =========================================================
model_data = joblib.load(f'model_sex{Gender}.pkl')
model = model_data['model']
scaler = model_data['scaler']
beta_model = model_data['beta_model']

# =========================================================
# Apply feature normalization using training statistics
# =========================================================
X_test_scaled = scaler.transform(X_test)

# =========================================================
# Generate age predictions on the held-out test set
# =========================================================
age_predic = model.predict(X_test_scaled)

# =========================================================
# Evaluate prediction accuracy
# =========================================================
r_predic = np.corrcoef(age_predic, age_test)[0, 1]
mae = mean_absolute_error(age_test, age_predic)
print(f'Prediction outcome: correlation r={r_predic:.2f}, MAE={mae:.2f}')

# =========================================================
# Compute brain age gap
# =========================================================
age_real = age_test
age_predic = age_predic
age_gap = age_predic - age_real

# =========================================================
# Correct age bias using parameters estimated from training data
# =========================================================
beta_coef = beta_model.coef_[0]
beta_intercept = beta_model.intercept_
age_fit = beta_coef * age_real + beta_intercept
gap_resid = age_gap - age_fit

# =========================================================
# Save test predictions and bias-corrected age gap
# =========================================================
joblib.dump(
    {
        'age_real': age_real,
        'age_predic': age_predic,
        'r_predic': r_predic,
        'mae': mae,
        'gap_resid': gap_resid
    },
    f'test_sex{Gender}.pkl'
)


## calculate age

In [ ]:
# =========================================================
# Model persistence and system-level utilities
# =========================================================
import joblib
import numpy as np
import os
import time
import psutil

# =========================================================
# Numerical computing and data handling
# =========================================================
import numpy as np
import pandas as pd

# =========================================================
# Machine learning models
# =========================================================
from sklearn.svm import LinearSVR

# =========================================================
# Model validation and performance evaluation
# =========================================================
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# =========================================================
# Feature preprocessing
# =========================================================
from sklearn.preprocessing import StandardScaler

# =========================================================
# Bias correction and statistical analysis
# =========================================================
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr

# =========================================================
# Parallel computation
# =========================================================
import joblib
from joblib import Parallel, delayed


In [ ]:
# =========================================================
# Model selection and loading
# =========================================================
Gender = 0  # Sex-stratified model selection (0: female, 1: male)

# Load pre-trained model components
model_data = joblib.load(f'model_sex{Gender}.pkl')

# Extract trained prediction and correction models
model = model_data['model']
scaler = model_data['scaler']
beta_model = model_data['beta_model']

# =========================================================
# Data input and feature extraction
# =========================================================
# Load new white-matter feature data
data = pd.read_csv('./input/white_matter_female_all.csv')

# Inspect available variables
print(data.columns)

# Construct feature matrix by excluding non-feature variables
X_new = data.drop(columns=['Unnamed: 0', 'eid', 'sex', 'age']).values

# =========================================================
# Prediction and bias correction
# =========================================================
# Apply training-derived feature normalization
X_new_scaled = scaler.transform(X_new)

# Generate brain age predictions
age_predic = model.predict(X_new_scaled)

# Apply linear age-bias correction
beta_coef = beta_model.coef_[0]
beta_intercept = beta_model.intercept_
age_fit = beta_coef * age_predic + beta_intercept
age_corrected = age_predic - age_fit

# =========================================================
# Output and data export
# =========================================================
# Report prediction results
print("Predicted Age:", age_predic)
print("Bias-Corrected Age:", age_corrected)

# Append prediction results to original dataset
data['Predicted_Age'] = age_predic
data['Bias_Corrected_Age'] = age_corrected

# Define output path and save results
output_directory = './output/'
output_filename = os.path.join(output_directory, 'white_matter_female.csv')
data.to_csv(output_filename, index=False)
